# Load and Process Data Prior to Feed

In [ ]:
import os
import numpy as np
import pandas as pd

# Data processing
from model.loader import *
from model.dataset import *
from model.preprocess import *
from model.augmentation import *

from model.visualization import *

File Accesses & Data Loader

In [ ]:
cd = os.getcwd()
print(cd)

set_path = cd + "data/processed/"     # Add this
access_folders = [set_path + "Input_set/", set_path + "Target_set/"]         # Input and Target

outpath = cd + "output/feed/"
processed_folders = [outpath + "Input_set/", outpath + "Target_set/"]        # Input and Target

In [ ]:
# Yo, might want to rename them files before ts
raw_input = load_folder(access_folders[0])
raw_target = load_folder(access_folders[1])

# Storing Structure: 
# Array of... dict1{"name", "shift", "intensity"} .....

Cleaning and Visuals

In [ ]:
# Brute-Force Spectrum Drag back, USE THIS IS BY CHOICE, NOT PER RECOMMENDATION
# Realign Shift Axis's Base Peak By Brute Force - For UNCALIBRATED samples
def brute_realign(shift, intensity):
    """
    Make sure the tallest peak visually is the baseline peak
    Loop this for the entire dataset if to apply
    """
    base_peak = np.max(intensity)
    base_loc = np.argwhere(base_peak)
    # Assigning Backwards
    for i in range(np.size(intensity) - base_loc):
        intensity[i] = intensity[base_loc]
    
    shift = shift[0:np.size(intensity) - base_loc]
    return shift, intensity

def group_realign(data):
    """Brute force fixture to crop the entire dataset, assume all dataset operates on the same shift scale"""
    # Brute-force realignment for every spectrum
    for spectrum in data:
        spectrum["shift"], spectrum["intensity"] = brute_realign(spectrum["shift"], spectrum["intensity"])
    # Find the one with the most data loss
    smallest = 3000
    for spectrum in data:
        array = spectrum["intensity"]
        if smallest > np.size(array):
            smallest = np.size(array)
    # Crop all data to the same size
    for spectrum in data:
        shift_axis = spectrum["shift"]
        inten_axis = spectrum["intensity"]
        low = np.min(shift_axis)
        high = shift_axis[smallest - 1]
        spectrum["shift"], spectrum["intensity"] = crop_region(shift_axis, inten_axis, low, high)

# Use this function alone or after brute force to tweak corrections for each data
def follow_correction(shift, intensity):        # Add more parameter as corrections needs
    """Add more filters as desired"""
    # Vector form normalization
    corr_shift = shift 
    corr_inten = normalize_vector(intensity)
    
    # Keep adding..
    #
    #
    #
    return corr_shift, corr_inten
    

In [ ]:
crop_input = raw_input
crop_target = raw_target
"""
# Brute-forcer: 
cropped_set = crop_input
cropped_set.append(crop_target)
cropped_set = group_crop(cropped_set)

# Validate and add correction
bad_crops = validate_lengths()     # Size check

# Re-distribute the crops
crop_input = cropped_set[0:len(crop_input) - 1]
crop_target = corppedd_set[len(crop_input):len(cropped_set) - 1]
"""
# Apply corrections
for spectrum in crop_input:
    spectrum["shift"], spectrum["intensity"] = follow_correction(spectrum["shift"], spectrum["intensity"])
for spectrum in crop_target:
    spectrum["shift"], spectrum["intensity"] = follow_correction(spectrum["shift"], spectrum["intensity"])

Boot Strapping and Splitting

In [ ]:
def bootstrap_gauss(dataset, variation=0.2, boot_size=200):
    # Find dataset's mean spectra intensities:
    num_points = len(dataset[0]["shift"])
    inten_sum = np.zeros(num_points)
    count = 0

    for spectrum in dataset:
        inten_sum += np.array(spectrum["intensity"])
        count += 1
    mean_inten = inten_sum / count
    
    boots = []
    for i in range(boot_size):
        boot_intensity = add_gaussian_noise(mean_inten, 0, variation)
        boot = {"filename": f"Bootstrap{i + 1}",
                "shift": dataset[0]["shift"],
                "intensity": boot_intensity}
        boots.append(boot)
    return boots

In [ ]:
# Append bootstrap to the mix
boot_input = crop_input + bootstrap_gauss(crop_input, 0.2, 100)
boot_target = crop_target + bootstrap_gauss(crop_target, 0.2, 100)

In [ ]:
# Pairing: 
# We might want to make a universal target spectrum to pair with
#

In [ ]:
# Splitting check dataset.py 

Output

In [ ]:
# Save each splitted dataset in a different folders, 3 for each, six total paths, use save_csv() for this
print(f"Saving to: {outpath}")

